In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, AIMessage, trim_messages, HumanMessage
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
load_dotenv()
from operator import itemgetter

model=ChatOpenAI(model='gpt-3.5-turbo')

trimmer=trim_messages(max_tokens=50, token_counter=model, include_system=True, allow_partial=False, start_on='human')

messages=[
    HumanMessage(content="Hello"),
    AIMessage(content="Hi there"),
    HumanMessage(content="tell me story about AI"),
    AIMessage(content="Once upon a time.... very long response..."),
    HumanMessage(content='can you summarize it'),
    AIMessage(content='sure i can do it so as i was saying that there was a time'),]

trimmer.invoke(messages)

parser=StrOutputParser()

prompt=ChatPromptTemplate.from_messages([('system','you are a helpful assistant and you have to remember the past conversation that we had about and on that basis you need to answer'),
                                  MessagesPlaceholder(variable_name='history'),('human','{input}')])

chain = (RunnablePassthrough.assign(history=itemgetter('history')|trimmer)|prompt|model|parser)
chain


RunnableAssign(mapper={
  history: RunnableLambda(itemgetter('history'))
           | RunnableLambda(...)
})
| ChatPromptTemplate(input_variables=['history', 'input'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[

In [4]:
chain.invoke({'history': messages, 'input':'on which topic i asked for a story'})

'You asked me to tell you a story about a helpful assistant, which is what inspired my previous response.'